## 2 Preprocessing

### Load libraries

In [1]:
library(tidyverse)
library(plotrix)
library(RColorBrewer)
library(Hmisc)
library(patchwork)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘Hmisc’


The following objects are masked from ‘package:dplyr’:

    src, summarize


The following objects are masked from ‘package:base’:

    format.pval, units




### Set working directory

In [ ]:
wd <- "/path/to/03_image_analysis/04_axenic_migration/"
setwd(wd)

### Collect and clean all CSV files exported by TrackMate

In [3]:
# Define folder containing raw CSVs
csv_folder_path <- file.path(wd, "02_data/trackmate/csv")

# List all CSV file paths
csv_paths <- list.files(csv_folder_path, pattern = "\\.csv$", full.names = TRUE)

# Read and clean each file into a named list
dataframes_list <- lapply(csv_paths, function(path) {
  base_name <- tools::file_path_sans_ext(basename(path))
  
  df <- read.csv(path, header = TRUE, check.names = TRUE, sep = ",")[-c(1:3), ] %>%
    dplyr::mutate(across(where(~ all(is.na(.) | is.numeric(.))), as.numeric)) %>%  # Convert numeric-like cols safely
    dplyr::mutate(sample = base_name)

  names(df) <- tolower(names(df))
  return(df)
})

# Name the list entries by the cleaned base filenames
names(dataframes_list) <- tools::file_path_sans_ext(basename(csv_paths))

Combine and clean TrackMate *spots* data

In [4]:
# Use grep to find names with "spots" in them
names_with_spots <- grep("spots", names(dataframes_list), value = TRUE)

# Subset the original list using the names with "spots"
spots_list <- dataframes_list[names_with_spots]

# Combine all dataframes in the list by rows
spots_df <- do.call(rbind, spots_list)

# Reset row names to make them sequential
rownames(spots_df) <- NULL

# Define the columns you want to remove
cols_to_remove <- c("label", "position_z", "position_t", "visibility", "manual_spot_color", "mean_intensity_ch1" , "median_intensity_ch1", "min_intensity_ch1",  "max_intensity_ch1", "total_intensity_ch1", "std_intensity_ch1", "contrast_ch1", "snr_ch1", "contrast_ch2", "snr_ch2")

### Processing data for the spots
Separate spot dataframes, clean up columns, and export as CSV.

In [17]:
# Extract spot-related tables
names_with_spots <- grep("spots", names(dataframes_list), value = TRUE)
spots_list <- dataframes_list[names_with_spots]

# Combine and clean spot data
spots_df <- do.call(rbind, spots_list)
rownames(spots_df) <- NULL

# Remove unnecessary metadata columns
cols_to_remove <- c(
  "label", "position_z", "position_t", "visibility", "manual_spot_color",
  "mean_intensity_ch1", "median_intensity_ch1", "min_intensity_ch1", 
  "max_intensity_ch1", "total_intensity_ch1", "std_intensity_ch1",
  "contrast_ch1", "snr_ch1", "contrast_ch2", "snr_ch2"
)
spots_df <- spots_df[, !(names(spots_df) %in% cols_to_remove)]

# Reorder and parse sample info
spots_df <- spots_df[, c("sample", setdiff(names(spots_df), "sample"))]
spots_df <- spots_df %>% separate(sample, into = c("series", NA), sep = "[_-]")

# Save cleaned spots dataframe
file_path <- paste0(wd, "/02_data/trackmate/csv_combined/spots.csv")
dir_path <- dirname(file_path)
if (!dir.exists(dir_path)) dir.create(dir_path, recursive = TRUE)
write.csv(spots_df, file_path, row.names = FALSE)


Warning message:
“Expected 2 pieces. Additional pieces discarded in 154959 rows [1, 2, 3, 4, 5,
6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, ...].”


### Processing data for the edges
Separate edge dataframes, combine, clean, and save as CSV

In [18]:
# Extract edge-related tables
names_with_edges <- grep("edges", names(dataframes_list), value = TRUE)
edges_list <- dataframes_list[names_with_edges]

# Combine and clean edge data
edges_df <- do.call(rbind, edges_list)
rownames(edges_df) <- NULL

# Remove unnecessary metadata columns
cols_to_remove <- c("label", "edge_z_location", "manual_edge_color")
edges_df <- edges_df[, !(names(edges_df) %in% cols_to_remove)]

# Reorder and parse sample info
edges_df <- edges_df[, c("sample", setdiff(names(edges_df), "sample"))]
edges_df <- edges_df %>% separate(sample, into = c("series", NA), sep = "_") 

# Save cleaned edges dataframe
file_path <- paste0(wd, "/02_data/trackmate/csv_combined/edges.csv")
dir_path <- dirname(file_path)
if (!dir.exists(dir_path)) dir.create(dir_path, recursive = TRUE)
write.csv(edges_df, file_path, row.names = FALSE)

### Processing data for the tracks
Separate dataframes of the tracks and combine them, perform some clean up and save as CSV

In [19]:
# Use grep to find names with "tracks" in them
names_with_tracks <- grep("tracks", names(dataframes_list), value = TRUE)

# Subset the original list using the names with "tracks"
tracks_list <- dataframes_list[names_with_tracks]

# Combine all dataframes in the list by rows
tracks_df <- do.call(rbind, tracks_list)

# Reset row names to make them sequential
rownames(tracks_df) <- NULL

# Define the columns you want to remove
cols_to_remove <- c("label", "track_index", "track_z_location")

# Remove the columns from the dataframe
tracks_df <- tracks_df[, !(names(tracks_df) %in% cols_to_remove)]

# Create a vector of column names with 'sample' first
column_order <- c("sample", setdiff(names(tracks_df), "sample"))

# Reorder the columns based on the vector
tracks_df <- tracks_df[, column_order]

# Separate the 'sample' column into 'exp' and 'strain'
tracks_df <- tracks_df %>% separate(sample, into = c("series", NA), sep = "_")

# Specify the path where you want to save the dataframe
file_path <- paste0(wd, "/02_data/trackmate/csv_combined/tracks.csv")

# Ensure the output directory exists
dir_path <- dirname(file_path)
if (!dir.exists(dir_path)) {
  dir.create(dir_path, recursive = TRUE)
}

# Save the cleaned tracks dataframe
write.csv(tracks_df, file_path, row.names = FALSE)